# DSO 522 | Machine Learning & Deep Learning for Forecasting
### A practical, intuition-first Google Colab workshop

**Goal:** Forecast daily store demand for the next **14 days**, compare models fairly, and learn when each method is useful.

**Run in order from the top.** All required data are generated in this notebook; no uploads or API keys are needed for the core exercises.

**Roadmap**
1. Understand the series and the forecasting horizon
2. Establish a seasonal-naive benchmark
3. Apply a familiar algorithm: XGBoost, now as a *forecaster*
4. Discover how an LSTM reads sequences
5. Use purpose-built forecasting networks: N-BEATS and NHITS
6. Compare forecasts and interpret business decisions
7. Optional introductions: Temporal Fusion Transformer and pretrained foundation models

> **Teaching point:** A more sophisticated algorithm is not necessarily a more accurate forecast. Test against a simple, honest benchmark.

## 0 | One-time setup
**Packages:** `numpy` and `pandas` organize the data, `matplotlib` draws plots, `scikit-learn` evaluates predictions, and `xgboost` provides gradient-boosted trees. `neuralforecast` is used for neural forecasting models.

**Colab:** Runtime → Run all. The neural section installs an additional library when you reach it. Neural training takes longer than the tree model, particularly on a CPU.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

SEED = 42  # Reproducibility: use the same random seed throughout.
H = 14     # Forecast horizon: predict the NEXT 14 days at once.
np.random.seed(SEED)
plt.rcParams.update({"figure.figsize": (11, 4.2), "axes.grid": True})
print("Ready. Forecast horizon:", H, "days")

## 1 | Business case: forecast store demand for the next 14 days

We use a **reproducible teaching dataset** (not real company data): daily sales depend on weekday patterns, trend, promotions, and temperature. The weather relationship is U-shaped—both hot and cold days can increase demand. This is useful because ML models can capture effects that simple linear correlation misses.

**Forecast origin:** the end of the training period. At that moment, we have **past sales**, the **future calendar**, **scheduled promotions**, and a **temperature forecast** (but NOT actual future temperatures or future sales). To keep the notebook self-contained, we simulate archived weather forecasts as a separate column. Forecasts here are imperfect proxies, not a weather service feed.

**Today:** train several models, predict the *same* 14 holdout days, visualize results, and discuss what a manager should do.

In [ ]:
rng = np.random.default_rng(SEED)
dates = pd.date_range("2024-01-01", periods=730, freq="D")
t = np.arange(len(dates))
weekday = dates.dayofweek
promotion = ((t % 31) >= 25).astype(int)  # Promotion schedule known in advance

# Forecast temperature is an imperfect, future-available planning input.
seasonal_temp = 65 + 18 * np.sin(2 * np.pi * (t - 100) / 365.25)
temp_forecast = seasonal_temp + rng.normal(0, 4, len(t))
temp_actual = temp_forecast + rng.normal(0, 5, len(t))  # Unknown future outcome

# Weather affects demand in BOTH extremes: cold and hot.
weather_effect = .45 * np.abs(temp_actual - 65)
sales = (140 + .055*t + 22*np.cos(2*np.pi*weekday/7)
         + 10*np.sin(2*np.pi*t/365.25) + 18*promotion
         + weather_effect + rng.normal(0, 8, len(t)))

series = pd.DataFrame({"ds": dates, "y": sales, "promotion": promotion,
                       "weekday": weekday, "temp_forecast": temp_forecast,
                       "temp_actual": temp_actual})
train = series.iloc[:-H].copy()
test = series.iloc[-H:].copy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.ds, series.y, color="#A8B4C6", lw=1, label="Daily demand")
ax.plot(train.ds.tail(90), train.y.tail(90), color="#194A78", lw=2, label="Observed history")
ax.axvline(train.ds.iloc[-1], color="#B85139", ls="--", label="Forecast origin")
ax.axvspan(test.ds.iloc[0], test.ds.iloc[-1], color="#F5D7CA", alpha=.45)
ax.set(title="Daily demand: all models forecast the SAME 14 days", ylabel="Units/day")
ax.legend(loc="upper left", ncol=2); plt.show()
print("Historical observations:", len(train), "| Future dates:", len(test))

### Visual intuition: why forecast temperature can matter

Demand may rise in both cold and hot weather. A near-zero **linear** correlation would not prove that weather is unimportant. Notice that we keep `temp_actual` only to understand how data were generated; **no model uses actual future temperature**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(train.temp_actual, train.y, s=8, alpha=.22)
axes[0].set(xlabel="Actual historical temperature (°F)", ylabel="Sales", title="Weather vs demand (historical exploration)")
axes[1].plot(test.ds, test.temp_forecast, "o-", label="Weather forecast (available)")
axes[1].plot(test.ds, test.temp_actual, "o--", label="Actual future weather (NOT available)")
axes[1].set(title="Future weather is uncertain", ylabel="Temperature (°F)")
axes[1].tick_params(axis="x", rotation=40)
axes[1].legend()
plt.tight_layout(); plt.show()

### A helpful zoom: weekly repetition
The weekly shape becomes clearer when we plot only the last eight weeks. Discuss: **What yesterday tells us versus what the same weekday last week tells us?**

In [ ]:
plt.plot(train.ds.tail(56), train.y.tail(56), marker=".", color="#194A78")
plt.title("Last 56 observed days: weekly rhythm + noise")
plt.xlabel("Date"); plt.ylabel("Units/day"); plt.show()

## 2 | Begin with a benchmark: seasonal naive
**Intuition:** “Next Monday will look like the most recent Monday.” For a 7-day seasonal cycle, repeat the last observed week.

**Parameters explained**
- `season_length=7`: number of observations per repeating seasonal cycle (7 days in a week).
- `H=14`: number of future days to predict.

This benchmark is inexpensive and often surprisingly difficult to beat. We judge every more complex method against it.

In [ ]:
season_length = 7
last_week = train.y.to_numpy()[-season_length:]
naive_pred = np.resize(last_week, H)

predictions = pd.DataFrame({"ds": test.ds.to_numpy(),
                            "actual": test.y.to_numpy(),
                            "Seasonal naive": naive_pred})

plt.plot(train.ds.tail(28), train.y.tail(28), label="Recent history", color="#194A78")
plt.plot(test.ds, test.y, "o-", label="Actual future (evaluation only)", color="black")
plt.plot(test.ds, naive_pred, "o--", label="Seasonal naive", color="#C37A32")
plt.title("Baseline forecast: repeat the previous week")
plt.ylabel("Demand"); plt.legend(); plt.show()

### Availability rule: an easy way to catch leakage

At the forecast origin, `train.y` is known; `test.y` is not. `test.promotion` is a preannounced schedule, and `test.temp_forecast` represents an archived weather outlook. `test.temp_actual` is **not available**. The forecasting code never reads the latter two unknown outcomes as predictors.

| Future date feature | Available? | Use in forecast? |
|---|---|---|
| Day of week, scheduled promotion | Yes | Yes |
| Weather forecast | Yes, if obtained at the origin | Yes |
| Actual future weather, actual future sales | No | **Never** |

One-minute exercise: Why is `series.y.rolling(7).mean()` invalid as a predictor for that same date, but `series.y.shift(1).rolling(7).mean()` valid for one-step training?

## 3 | XGBoost: familiar ML, new forecasting workflow
You already know gradient-boosted trees. **What changes in forecasting?** We have to turn history into features and never leak unknown future demand into prediction.

**Lag 1** = yesterday's demand; **Lag 7** = demand on the same weekday one week ago; **Lag 14** = two weeks ago.

We also use the **weekday** and a **planned promotion flag**, both available for future dates.

For simplicity, we train a **one-day-ahead model** and use it **recursively** for 14 future days. Day 2 uses the model's prediction for Day 1 when a lag requires that observation.

**Trade-off:** recursion is easy to implement but errors can accumulate over longer horizons.

In [ ]:
LAGS = [1, 7, 14, 28]  # Yesterday, same day last week, two and four weeks ago

def make_lagged_training_data(history, include_business=False):
    """Make training examples using only the sales known BEFORE each day.

    history: training-period rows only (never the future test rows)
    include_business: add the calendar, planned promotion, and WEATHER FORECAST
    """
    frame = history[["ds", "y", "weekday", "promotion", "temp_forecast"]].copy()
    for lag in LAGS:
        frame[f"lag_{lag}"] = frame["y"].shift(lag)
    frame["rolling_7"] = frame["y"].shift(1).rolling(7).mean()  # shift first!
    frame = frame.dropna()
    features = [f"lag_{lag}" for lag in LAGS] + ["rolling_7"]
    if include_business:
        features += ["weekday", "promotion", "temp_forecast"]
    return frame[features], frame["y"], features

X_history, y_history, HISTORY_FEATURES = make_lagged_training_data(train)
X_business, y_business, BUSINESS_FEATURES = make_lagged_training_data(train, True)
print("History-only predictors:", HISTORY_FEATURES)
print("With business inputs:", BUSINESS_FEATURES)
display(pd.concat([X_business.head(3), y_business.head(3).rename("target_y")], axis=1))

### Train two XGBoost forecasters: fair architecture comparison vs. extra business information

We deliberately fit **two versions of the same algorithm**: (a) only past sales and (b) past sales + future-known features. The difference measures the potential value of business information, not the superiority of one architecture.

- `n_estimators=250`: number of boosting trees (more trees increase learning capacity and time).
- `max_depth=3`: maximum tree complexity (large values can overfit).
- `learning_rate=0.05`: how much each new tree adjusts the forecast.
- `random_state=42`: reproducible random choices.

**Forecasting method:** recursive. For every future day we recalculate lags and the shifted rolling average using historical sales or earlier predictions—not actual future sales.

In [ ]:
def fit_and_forecast_xgb(include_business=False):
    """Fit XGBoost on history, then recursively forecast the next H days.

    include_business chooses whether planned promotions and forecast weather
    are allowed; NEVER reads test['y'] or test['temp_actual'].
    """
    X, y, feature_names = make_lagged_training_data(train, include_business)
    model = XGBRegressor(n_estimators=250, max_depth=3, learning_rate=0.05,
                         objective="reg:squarederror", random_state=SEED, n_jobs=2)
    model.fit(X, y)
    history = list(train.y.to_numpy())
    future_predictions = []
    for date in test.itertuples(index=False):
        next_inputs = {f"lag_{lag}": history[-lag] for lag in LAGS}
        next_inputs["rolling_7"] = np.mean(history[-7:])
        if include_business:
            next_inputs.update(weekday=date.weekday, promotion=date.promotion,
                               temp_forecast=date.temp_forecast)
        prediction = float(model.predict(pd.DataFrame([next_inputs])[feature_names])[0])
        future_predictions.append(prediction)
        history.append(prediction)
    return model, np.array(future_predictions), feature_names

xgb_history, pred_history, _ = fit_and_forecast_xgb(False)
xgb_business, pred_business, FEATURES = fit_and_forecast_xgb(True)
predictions["XGBoost"] = pred_history  # fairer comparison: history-only
predictions["XGBoost + business"] = pred_business

plt.plot(test.ds, test.y, "o-", color="black", label="Actual")
plt.plot(test.ds, pred_history, "o--", label="XGBoost: history only")
plt.plot(test.ds, pred_business, "o--", label="XGBoost + planned inputs")
plt.title("What do scheduled promotions and forecast weather add?")
plt.ylabel("Units/day"); plt.legend(); plt.show()

### Feature importance—and the more reliable ablation test

**Feature importance** ranks how a fitted model uses inputs; it does *not* prove causation. An **ablation** is more directly actionable: compare forecasts with and without future-known business information on exactly the same dates. A single holdout is illustrative, not definitive.

In [ ]:
importance = pd.Series(xgb_business.feature_importances_, index=FEATURES).sort_values()
importance.plot(kind="barh", color="#247A78")
plt.title("XGBoost + business inputs: relative feature importance")
plt.xlabel("Model-specific importance (not causation)"); plt.show()

for label, predicted in [("Historical predictors", pred_history),
                         ("+ known future business inputs", pred_business)]:
    print(f"{label}: MAE = {mean_absolute_error(test.y, predicted):.2f} units")

### Mini backtest: repeat the forecast at several historical cutoffs

One unusually good or bad fortnight can distort comparisons. In this short demonstration we re-fit a **history-only XGBoost** at three earlier cutoffs and compare it with seasonal naive. This is a teaching-sized rolling-origin check, not a full production validation.

In [ ]:
backtest_rows = []
for cutoff in [len(series)-H*4, len(series)-H*3, len(series)-H*2]:
    past = series.iloc[:cutoff].copy()
    future = series.iloc[cutoff:cutoff+H].copy()
    x, y, f = make_lagged_training_data(past, False)
    m = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=.08,
                     objective="reg:squarederror", random_state=SEED, n_jobs=2)
    m.fit(x, y)
    history = list(past.y)
    forecast = []
    for _ in range(H):
        row = {f"lag_{lag}": history[-lag] for lag in LAGS}
        row["rolling_7"] = np.mean(history[-7:])
        p = float(m.predict(pd.DataFrame([row])[f])[0])
        forecast.append(p)
        history.append(p)
    naive = np.resize(past.y.to_numpy()[-7:], H)
    backtest_rows.append({"Origin": past.ds.iloc[-1].date(),
                          "Naive MAE": mean_absolute_error(future.y, naive),
                          "XGBoost MAE": mean_absolute_error(future.y, forecast)})
backtest = pd.DataFrame(backtest_rows).set_index("Origin")
display(backtest.round(2))
backtest.plot.bar(figsize=(9, 4), title="MAE varies across forecast origins (lower is better)")
plt.ylabel("MAE (units/day)"); plt.xticks(rotation=0); plt.show()

## 4 | LSTM: how a neural network reads an ordered sequence
**Intuition:** An LSTM reads observations sequentially and updates an internal state—loosely, a memory of what it finds useful. The network learns nonlinear patterns rather than relying exclusively on explicitly specified lags.

**Why it might help:** longer or complex temporal dependence. **Why it might not:** tuning, scaling, and data requirements can outweigh benefits on small business series.

We train a compact LSTM to predict **one next-day value** from the previous 28 days, then forecast recursively for 14 days. To keep this lesson clear, it uses sales only, so it does **not** receive the future promotion information XGBoost receives. That limits any head-to-head interpretation.

**Parameters**
- `lookback=28`: the number of past days the network sees.
- `LSTM(24)`: 24 internal learned units (model capacity).
- `epochs=12`: 12 passes through training windows.
- `batch_size=32`: number of windows processed per training update.

First visualize the windows. Each row contains 28 days of history; its target is the following day.

In [ ]:
LOOKBACK = 28
values = train.y.to_numpy(dtype="float32")
mean_y = float(values.mean()); std_y = float(values.std())
scaled = (values - mean_y) / std_y  # Scale using TRAIN ONLY.

def sequence_windows(a, lookback):
    """Build (past window, next value) pairs without future data leakage."""
    x = np.array([a[i-lookback:i] for i in range(lookback, len(a))])
    y = np.array([a[i] for i in range(lookback, len(a))])
    return x[..., np.newaxis], y

X_seq, y_seq = sequence_windows(scaled, LOOKBACK)
print("Training sequence tensor:", X_seq.shape,
      "= (windows, days per window, features per day)")
plt.plot(np.arange(LOOKBACK), X_seq[0, :, 0], marker="o", label="Historical input")
plt.scatter([LOOKBACK], [y_seq[0]], color="#B85139", s=90, label="Target: next day")
plt.title("One training example for LSTM (scaled demand)")
plt.xlabel("Position in historical window"); plt.legend(); plt.show()

### Fit LSTM with Keras (TensorFlow)
This section installs TensorFlow only if unavailable. Training on a CPU is usually feasible for this small teaching network. Results can differ slightly by hardware and TensorFlow version even with a seed.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("tensorflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "tensorflow"])
import tensorflow as tf
from tensorflow import keras
tf.keras.utils.set_random_seed(SEED)

lstm = keras.Sequential([
    keras.layers.Input(shape=(LOOKBACK, 1)),
    keras.layers.LSTM(24),
    keras.layers.Dense(1)
])
lstm.compile(optimizer="adam", loss="mse")
fit_history = lstm.fit(X_seq, y_seq, epochs=12, batch_size=32,
                       shuffle=False, verbose=0)
plt.plot(fit_history.history["loss"], marker="o")
plt.title("LSTM training loss: lower means a better fit to training windows")
plt.xlabel("Epoch"); plt.ylabel("Mean squared error (scaled)"); plt.show()

In [ ]:
window = list(scaled[-LOOKBACK:])
lstm_future_scaled = []
for _ in range(H):
    x = np.asarray(window[-LOOKBACK:], dtype="float32").reshape(1, LOOKBACK, 1)
    p = float(lstm.predict(x, verbose=0)[0, 0])
    lstm_future_scaled.append(p)
    window.append(p)

predictions["LSTM"] = np.asarray(lstm_future_scaled) * std_y + mean_y
plt.plot(test.ds, test.y, "o-", color="black", label="Actual")
plt.plot(test.ds, predictions["LSTM"], "o--", label="LSTM", color="#8867A4")
plt.title("LSTM: 14 days recursively predicted from 28-day windows")
plt.ylabel("Units/day"); plt.legend(); plt.show()

## 5 | N-BEATS and NHITS: networks built specifically for forecasting
Both models take a **past window** and output a **whole future horizon** at once, unlike the one-step recursive XGBoost/LSTM examples above.

### N-BEATS — iterative forecast refinement
**Intuition:** Feedforward blocks learn forecasting patterns and successively refine a forecast. Unlike an LSTM, it does not process each historical day one by one through a recurrent memory state.

### NHITS — look at multiple time scales
**Intuition:** A network may benefit from seeing both broad changes and short-term details. NHITS uses hierarchical, multi-resolution representations and interpolation.

**Important shared parameters**
- `h=14`: produce 14 days of forecasts together.
- `input_size=56`: look at the last 56 observed days for each training example.
- `max_steps=100`: number of optimization updates; smaller = quicker demo, potentially less accurate.
- `random_seed=42`: controls part of the random initialization/training behavior.

These are simple **univariate** examples. Both models have additional tunable architecture options; we leave defaults in place so students focus on forecasting rather than network engineering.

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("neuralforecast") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "neuralforecast"])
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS

nf = NeuralForecast(
    models=[
        NBEATS(h=H, input_size=56, max_steps=100, random_seed=SEED),
        NHITS(h=H, input_size=56, max_steps=100, random_seed=SEED)
    ],
    freq="D"  # One observation per calendar day.
)

nf_train = train[["ds", "y"]].copy()
nf_train.insert(0, "unique_id", "Store_A")
nf.fit(df=nf_train)
deep_future = nf.predict().sort_values("ds").reset_index(drop=True)
display(deep_future.head())

In [ ]:
# Both forecasts should have exactly H dates, and must match our holdout.
assert len(deep_future) == H
assert np.array_equal(pd.to_datetime(deep_future.ds).to_numpy(), test.ds.to_numpy())
predictions["N-BEATS"] = deep_future["NBEATS"].to_numpy()
predictions["NHITS"] = deep_future["NHITS"].to_numpy()

plt.plot(test.ds, test.y, "o-", color="black", lw=2, label="Actual")
plt.plot(test.ds, predictions["N-BEATS"], "o--", label="N-BEATS")
plt.plot(test.ds, predictions["NHITS"], "o--", label="NHITS")
plt.title("Specialized neural models: direct 14-day forecasts")
plt.ylabel("Units/day"); plt.legend(); plt.show()

## 6 | Compare forecasting approaches without overstating the conclusions

All models predict the **same 14 dates from the same historical cutoff**. The seasonal naive, history-only XGBoost, LSTM, N-BEATS and NHITS use historical demand only. The additional **XGBoost + business** model uses planned promotions and archived simulated temperature forecasts: its improvement, if any, is partly an **information advantage**.

Use **MAE** (typical absolute mistake) and **RMSE** (emphasizes unusually large mistakes). This is only a single teaching holdout. Real model selection should validate across several forecast origins and business regimes.

In [ ]:
model_names = ["Seasonal naive", "XGBoost", "XGBoost + business",
               "LSTM", "N-BEATS", "NHITS"]
score_rows = []
for name in model_names:
    score_rows.append({
        "Model": name,
        "MAE (units)": mean_absolute_error(predictions.actual, predictions[name]),
        "RMSE (units)": np.sqrt(mean_squared_error(predictions.actual, predictions[name]))
    })
scores = pd.DataFrame(score_rows).sort_values("MAE (units)")
display(scores.style.format({"MAE (units)": "{:.2f}", "RMSE (units)": "{:.2f}"}))

ax = scores.sort_values("MAE (units)").plot.barh(
    x="Model", y="MAE (units)", legend=False, color="#247A78")
ax.invert_yaxis(); ax.set_xlabel("MAE (units/day; lower is better)")
ax.set_title("Illustrative single-holdout accuracy — not a definitive ranking")
plt.show()
print("IMPORTANT: The business-informed XGBoost uses extra FUTURE-KNOWN variables;")
print("the other models use only historical demand. Do not interpret the")
print("difference as a controlled architecture contest.")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(train.ds.tail(42), train.y.tail(42), color="#9AA8BC", label="History")
plt.plot(predictions.ds, predictions.actual, "o-", color="black", lw=2.5, label="Actual holdout")
for name in model_names:
    plt.plot(predictions.ds, predictions[name], marker=".", alpha=.85, label=name)
plt.axvline(train.ds.iloc[-1], ls="--", color="#B85139")
plt.title("Same dates and forecast origin; different input availability")
plt.ylabel("Demand (units/day)"); plt.xlabel("Date")
plt.legend(ncol=3, fontsize=9); plt.show()

### Error by forecast day: do mistakes grow with the horizon?

Recursive methods can accumulate error; a single aggregate MAE hides which days are difficult. Plot absolute error by lead time, from day 1 through day 14.

In [ ]:
plt.figure(figsize=(10, 4))
for name in ["Seasonal naive", "XGBoost", "XGBoost + business", "NHITS"]:
    err = np.abs(predictions.actual.to_numpy() - predictions[name].to_numpy())
    plt.plot(np.arange(1, H+1), err, marker="o", label=name)
plt.xlabel("Days ahead"); plt.ylabel("Absolute error (units/day)")
plt.title("Errors at each forecast horizon — one holdout origin")
plt.legend(); plt.show()

### Operational interpretation: shortages and excess inventory

Suppose a store orders exactly the predicted demand each day. **Under-forecasting** produces shortages; **over-forecasting** produces leftover inventory. These two errors may have different dollar costs. In reality, forecasts inform ordering decisions, rather than mechanically determining the exact order size. The example below illustrates the trade-off.

In [ ]:
under, over = {}, {}
for name in model_names:
    error = predictions.actual.to_numpy() - predictions[name].to_numpy()
    under[name] = np.maximum(error, 0).sum()
    over[name] = np.maximum(-error, 0).sum()
costs = pd.DataFrame({"Underforecast units": under, "Overforecast units": over})
costs.plot(kind="bar", figsize=(11, 4), color=["#B85139", "#247A78"])
plt.title("Which kinds of mistakes would the business face?")
plt.ylabel("Total error units, 14-day holdout"); plt.xticks(rotation=20); plt.show()

shortage_cost = 12  # illustrative dollars per unmet unit
surplus_cost = 3    # illustrative dollars per excess unit
costs["Illustrative $ cost"] = shortage_cost * costs["Underforecast units"] + surplus_cost * costs["Overforecast units"]
display(costs.sort_values("Illustrative $ cost").round(1))

## 7 | TFT: a Transformer that can use business context (optional instructor demonstration)
**Temporal Fusion Transformer (TFT)** brings together several types of inputs:
- **Static:** store location, product category (does not change during the forecast).
- **Historical:** past actual sales or observed weather (known only up to forecast time).
- **Future-known:** the weekday, holiday schedule, or planned promotion for each future date.

Its attention-based architecture helps the network combine temporal signals. However, attention is **not** a reliable causal explanation.

**When is this attractive?** Many related series with rich contextual data and enough observations to train a more complex model. **Cost:** more setup, tuning, and computation.

### Optional API illustration (not part of the timed core lab)
The model declaration below is valid as a conceptual starting point. Fitting a TFT with covariates also requires organizing the `futr_exog_list` columns for **both** historical training and future prediction data. We intentionally do not fit TFT here: without these steps, a short demo would risk silently using information unavailable at prediction time.

In [ ]:
from neuralforecast.models import TFT

# Read the parameters; this cell only creates a model object.
# h: 14 forecast days; input_size: 56 observed days.
# max_steps: training updates; hidden_size: network capacity.
tft_example = TFT(h=H, input_size=56, hidden_size=16,
                  max_steps=50, random_seed=SEED,
                  futr_exog_list=["promotion", "weekday"])
print("TFT configuration prepared. Not trained in this optional example.")

## 8 | Foundation models: a pretrained forecasting starting point (optional)
**Intuition:** A forecasting foundation model has already been pretrained on many time series. You can supply a new historical series and request a forecast without training a dedicated neural network for that series. This is called **zero-shot** forecasting.

Examples include **Chronos** (Amazon) and **TimesFM** (Google). Version, model size, runtime, and license matter. A zero-shot forecast is a useful benchmark, **not a guarantee of accuracy**.

The following **optional code** loads pretrained weights from Hugging Face; it needs internet access and can require substantial memory and download time. For a live lecture, consider running it before class. It is NOT needed for the core model comparison.

In [ ]:
# OPTIONAL: pretrained Chronos-Bolt, requires an internet download.
# Run this cell in Colab only when you want a zero-shot demonstration.
# %pip -q install chronos-forecasting
# import torch
# from chronos import BaseChronosPipeline
# pipe = BaseChronosPipeline.from_pretrained(
#     "amazon/chronos-bolt-small", device_map="cpu", torch_dtype=torch.float32
# )
# q, mean = pipe.predict_quantiles(
#     context=torch.tensor(train.y.to_numpy(), dtype=torch.float32),
#     prediction_length=H, quantile_levels=[0.1, 0.5, 0.9]
# )
# p10, median, p90 = [q[0, :, i].numpy() for i in range(3)]
# plt.plot(test.ds, median, "o-", label="Chronos P50")
# plt.fill_between(test.ds, p10, p90, alpha=.2, label="P10–P90 forecast interval")
# plt.plot(test.ds, test.y, "k.-", label="Actual")
# plt.legend(); plt.title("Zero-shot pretrained forecast"); plt.show()
print("Optional Chronos zero-shot demonstration is OFF by default.")

## 9 | Takeaways and quick class exercises

**Choose a model by the forecasting problem—not by how modern it sounds.**

| Situation | Reasonable starting models |
|---|---|
| Small, stable seasonal series | Seasonal naive, ETS, ARIMA |
| Many engineered lags + known business factors | XGBoost / LightGBM |
| Learn patterns in ordered history | LSTM |
| Pure historical-series forecasting | N-BEATS |
| Multi-scale / longer-horizon patterns | NHITS |
| Many series with rich known-future predictors | TFT |
| New series, want a quick benchmark without fitting | Chronos / TimesFM |

**Exercises (do not need to retrain everything):**
1. Change `LAGS` to `[1, 7, 14]`. How does XGBoost performance change? Why might it change?
2. Change `H` from 14 to 7, rerun from the top, and compare the models. (A horizon change affects the train/test cutoff.)
3. Explain why “tomorrow's actual demand” cannot be used to construct features for a forecast made today.
4. Is a 14-day holdout enough to select a production model? Explain how rolling-origin evaluation would improve confidence.

**Further reading / APIs**
- NeuralForecast: https://nixtlaverse.nixtla.io/neuralforecast/
- XGBoost Python: https://xgboost.readthedocs.io/
- Chronos foundation models: https://chronos-ts.ai/
- scikit-learn time-related feature engineering: https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html

**Instructor note:** This is a single-origin teaching exercise, not a model benchmark or causal analysis. For rigorous model ranking, use rolling origins, comparable feature access, tuning on separate validation windows, and repeated experiments.

**Instructor caveats:** Data are simulated; the weather outlook is not a genuine archived vendor forecast. A common holdout does not eliminate the extra-feature advantage of business-informed XGBoost. Model selection would require several rolling origins, appropriate future covariates for each architecture, and resource/runtime checks. On a fresh Colab runtime, neural packages may take time to install or train.